# 00 - One-time setup

Run once, before the first collection. Creates the schemas `landing`, `bronze`, `silver`, `gold`, `ops` (silver and gold stay empty), the staging volume and its folders. Safe to run again.

In [ ]:
# ONE-TIME SETUP - creates the five schemas (landing, bronze, silver, gold, ops), the staging volume and its folders.
# Run it once before the first full load. It is safe to run again (everything is IF NOT EXISTS).
# The Bronze notebook would also create bronze and ops by itself; silver and gold stay empty until they are built.

dbutils.widgets.text("catalog", "workspace", "catalog")
CATALOG = dbutils.widgets.get("catalog").strip()
ROOT = f"/Volumes/{CATALOG}/landing/staging/frontier"

# Medallion schemas. landing = staging area, bronze = raw records, ops = logs.
# silver and gold are created now but stay EMPTY: they are reserved for the next phases.
for schema in ["landing", "bronze", "silver", "gold", "ops"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.landing.staging")
for folder in ["new", "new/_manifests", "processed", "processed/_manifests", "quarantine", "_state",
               "logs/pending", "logs/ingested"]:
    dbutils.fs.mkdirs(f"{ROOT}/{folder}")

print("Volume ready:", ROOT)
display(dbutils.fs.ls(ROOT))
display(spark.sql(f"SHOW SCHEMAS IN {CATALOG}"))